# 1) MLP: A * B % C (A,B between 0..99, C between 0..19)

In [1]:
#@title (0) Setup
from google.colab import drive
import sys
drive.mount('/content/drive')
#sys.path.append('/content/drive/MyDrive/Colab Notebooks')

%run "/content/drive/MyDrive/Colab Notebooks/Build your own LLM/han-common.ipynb"

Mounted at /content/drive
Train: 152,000 samples  |  Test: 38,000 samples
Input dim: 219  |  Output classes: 19
Label distribution (train): {np.int64(0): np.int64(51411), np.int64(1): np.int64(11977), np.int64(2): np.int64(14363), np.int64(3): np.int64(10413), np.int64(4): np.int64(11615), np.int64(5): np.int64(6717), np.int64(6): np.int64(10053), np.int64(7): np.int64(4823), np.int64(8): np.int64(6869), np.int64(9): np.int64(4650), np.int64(10): np.int64(4769), np.int64(11): np.int64(2542), np.int64(12): np.int64(4452), np.int64(13): np.int64(1747), np.int64(14): np.int64(2023), np.int64(15): np.int64(1374), np.int64(16): np.int64(1254), np.int64(17): np.int64(538), np.int64(18): np.int64(410)}
DNN(
  (proj_in): Linear(in_features=219, out_features=128, bias=True)
  (layers): ModuleList(
    (0-15): 16 x Sequential(
      (0): Linear(in_features=128, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=128, bias=True)
    )
  )
  (norms): Modu

## 1.2) DNN: 219-> 32 x 256 w/ ReLU+Residuals ->19

In [ ]:
class DNN(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, output_dim=OUTPUT_DIM, hidden_dim=256,
                 layer_depth=32, has_residuals=False):
        super().__init__()
        self.has_residuals = has_residuals
        self.proj_in = nn.Linear(input_dim, hidden_dim)
        self.layers = nn.ModuleList()
        for _ in range(layer_depth):
            self.layers.append(nn.Linear(hidden_dim, hidden_dim))
        self.proj_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = F.relu(self.proj_in(x))
        for layer in self.layers:
            if self.has_residuals:
                x = x + F.relu(layer(x))
            else:
                x = F.relu(layer(x))
        return self.proj_out(x)

MODEL = DNN().to(DEVICE)
print(MODEL)

print(f"Total parameters: {sum(p.numel() for p in MODEL.parameters()):,}")
print(f"Per-layer shapes:")
for name, param in MODEL.named_parameters():
    if "weight" in name:
        print(f"  {name:30s}  {str(list(param.shape)):>15s}  ({param.numel():>6,} params)")


DNN(
  (proj_in): Linear(in_features=219, out_features=256, bias=True)
  (layers): ModuleList(
    (0-31): 32 x Linear(in_features=256, out_features=256, bias=True)
  )
  (proj_out): Linear(in_features=256, out_features=19, bias=True)
)
Total parameters: 2,166,547
Per-layer shapes:
  proj_in.weight                       [256, 219]  (56,064 params)
  layers.0.weight                      [256, 256]  (65,536 params)
  layers.1.weight                      [256, 256]  (65,536 params)
  layers.2.weight                      [256, 256]  (65,536 params)
  layers.3.weight                      [256, 256]  (65,536 params)
  layers.4.weight                      [256, 256]  (65,536 params)
  layers.5.weight                      [256, 256]  (65,536 params)
  layers.6.weight                      [256, 256]  (65,536 params)
  layers.7.weight                      [256, 256]  (65,536 params)
  layers.8.weight                      [256, 256]  (65,536 params)
  layers.9.weight                      [256, 25

## 1.3) Train



In [2]:
def train(run_name="", layer_depth=8, has_residuals=False):
    model = DNN(layer_depth=layer_depth, has_residuals=has_residuals).to(DEVICE)
    return train_common(model, run_name)

# 2) 32 Layers: No Residuals


In [ ]:
MODEL = train(run_name="32 layers, no residuals", layer_depth=32, has_residuals=False)

  🚀 Run: 32 layers, no residuals


/usr/local/lib/python3.13/dist-packages/torch/_inductor/compile_fx.py:322: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
W1020 10:45:01.265000 1817 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


Epoch      1 | Train loss 1.7763  acc 37.8% | Test loss 1.7788  acc 37.8%
Epoch      2 | Train loss 1.5465  acc 45.3% | Test loss 1.5541  acc 45.2%
Epoch      3 | Train loss 1.6083  acc 44.0% | Test loss 1.6198  acc 43.7%
Epoch      4 | Train loss 1.6005  acc 45.1% | Test loss 1.6099  acc 44.8%
Epoch      5 | Train loss 1.3943  acc 48.5% | Test loss 1.4004  acc 48.3%
Epoch      6 | Train loss 1.3738  acc 49.2% | Test loss 1.3827  acc 49.3%
Epoch      7 | Train loss 1.3922  acc 49.8% | Test loss 1.4009  acc 49.4%
Epoch      8 | Train loss 1.5219  acc 46.2% | Test loss 1.5311  acc 46.1%
Epoch      9 | Train loss 1.5092  acc 46.7% | Test loss 1.5157  acc 46.5%
Epoch     10 | Train loss 1.4425  acc 47.4% | Test loss 1.4459  acc 47.1%
Epoch     11 | Train loss 1.4307  acc 46.9% | Test loss 1.4391  acc 46.6%
Epoch     12 | Train loss 1.3716  acc 49.5% | Test loss 1.3768  acc 49.3%
Epoch     13 | Train loss 1.3585  acc 50.1% | Test loss 1.3659  acc 50.1%
Epoch     14 | Train loss 1.4572  acc 

# 3) 32 Layers: With Residuals

In [ ]:
MODEL = train(run_name="32 layers, residuals", layer_depth=32, has_residuals=True)

  🚀 Run: 32 layers, residuals
Epoch      1 | Train loss 76.9373  acc 20.2% | Test loss 80.3121  acc 20.2%
Epoch      2 | Train loss 21.5624  acc 21.4% | Test loss 22.4418  acc 21.1%
Epoch      3 | Train loss 4.2248  acc 23.2% | Test loss 4.2753  acc 23.2%
Epoch      4 | Train loss 3.5304  acc 27.4% | Test loss 3.5793  acc 27.3%
Epoch      5 | Train loss 3.1797  acc 27.8% | Test loss 3.2303  acc 27.4%
Epoch      6 | Train loss 2.9974  acc 27.4% | Test loss 3.0520  acc 26.9%
Epoch      7 | Train loss 2.7358  acc 27.8% | Test loss 2.7948  acc 27.4%
Epoch      8 | Train loss 2.6015  acc 29.4% | Test loss 2.6552  acc 28.8%
Epoch      9 | Train loss 2.4827  acc 31.3% | Test loss 2.5333  acc 30.7%
Epoch     10 | Train loss 2.5697  acc 29.6% | Test loss 2.6288  acc 28.6%
Epoch     11 | Train loss 2.4828  acc 32.8% | Test loss 2.5639  acc 31.8%
Epoch     12 | Train loss 2.2748  acc 28.4% | Test loss 2.3446  acc 27.4%
Epoch     13 | Train loss 2.2820  acc 33.0% | Test loss 2.3656  acc 31.8%
Epoc

# 4) Loss + Accuracy across runs

In [ ]:
#@title Visualize
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.colors as mcolors
import logging
import matplotlib.ticker as ticker

# Suppress missing font warnings
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

# Prefer the deployment font when available, otherwise fall back to common sans-serif fonts
font_candidates = ['Inter', 'DejaVu Sans', 'Arial', 'Helvetica', 'sans-serif']
plt.rcParams['font.family'] = font_candidates
font_path = 'fonts/Inter-Regular.ttf'
try:
    prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = prop.get_name()
except Exception:
    pass

# Set styling for all text and lines
plt.rcParams['text.color'] = '#F1F5F9'
plt.rcParams['axes.labelcolor'] = '#F1F5F9'
plt.rcParams['xtick.color'] = '#F1F5F9'
plt.rcParams['ytick.color'] = '#F1F5F9'
plt.rcParams['axes.edgecolor'] = '#F1F5F9'

# Grid styling
plt.rcParams['grid.color'] = '#5B7268'
plt.rcParams['grid.alpha'] = 0.7

def _coplot_luminance_from_rgba(rgba):
    r, g, b = mcolors.to_rgb(rgba)
    def _linear(c):
        return c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4
    return 0.2126 * _linear(r) + 0.7152 * _linear(g) + 0.0722 * _linear(b)

def _coplot_text_color_for_rgba(rgba, preferred='#F1F5F9'):
    try:
        preferred_l = _coplot_luminance_from_rgba(mcolors.to_rgba(preferred))
        bg_l = _coplot_luminance_from_rgba(rgba)
        lighter = max(preferred_l, bg_l)
        darker = min(preferred_l, bg_l)
        ratio = (lighter + 0.05) / (darker + 0.05)
        if ratio >= 4.5:
            return preferred
    except Exception:
        pass
    bg_l = _coplot_luminance_from_rgba(rgba)
    return '#111111' if bg_l > 0.5 else '#FFFFFF'

# Create figure and axis with publication-quality settings
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300)
fig.patch.set_facecolor('#1A3A2A')

# Distinct bright colors for dark background (modified to keep runs distinguishable)
colors = ["#38BDF8", "#34D399", "#FBBF24", "#F87171", "#A78BFA"]
line_styles = ["-","--",":","-."]
marker_styles = ["o","s","^","D"]

for idx, run in enumerate(all_runs):
    h = run["history"]

    line_style = line_styles[idx % len(line_styles)] if False else '-'
    marker_style = marker_styles[idx % len(marker_styles)] if False else ''

    # Plot accuracy
    axes[0].plot(h["epoch"], h["test_acc"], label=run["name"],
                 color=colors[idx % len(colors)], linewidth=2,
                 linestyle=line_style, marker=marker_style, alpha=0.9)

    # Plot loss
    axes[1].plot(h["epoch"], h["test_loss"], label=run["name"],
                 color=colors[idx % len(colors)], linewidth=2,
                 linestyle=line_style, marker=marker_style, alpha=0.9)

# Apply custom styling to both axes
for ax in axes:
    ax.set_facecolor('#1A3A2A')
    ax.set_xlabel('Epoch', fontsize=12, labelpad=10, color='#F1F5F9')

    # Force integer epochs
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

    # Add grid
    ax.grid(axis='both', alpha=0.7, color='#5B7268', linestyle='--', linewidth=0.5, zorder=0)
    ax.set_axisbelow(True)

    # Remove top and right spines for cleaner look
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(1)
    ax.spines['left'].set_linewidth(1)
    ax.spines['bottom'].set_color('#F1F5F9')
    ax.spines['left'].set_color('#F1F5F9')

    # Adjust spine positions
    ax.spines['left'].set_position(('outward', 2))
    ax.spines['bottom'].set_position(('outward', 2))

    # Add legend
    ax.legend(frameon=False, fontsize=10)

# Specific titles and labels
axes[0].set_ylabel('Test Accuracy', fontsize=12, color='#F1F5F9', labelpad=10)
axes[0].set_title('Test Accuracy by Init Method', color='#F1F5F9', pad=15)
axes[0].yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1.0))

axes[1].set_ylabel('Test Loss', fontsize=12, color='#F1F5F9', labelpad=10)
axes[1].set_title('Test Loss by Init Method', color='#F1F5F9', pad=15)
axes[1].set_yscale('linear')
axes[1].set_ylim(top=3.0)
axes[1].set_ylim(bottom=0.0)
axes[1].yaxis.set_major_formatter(ticker.ScalarFormatter())

# Global title
plt.suptitle("Initialization Method Comparison", fontsize=14, y=1.02, color='#F1F5F9')

# Tight layout
plt.tight_layout()
plt.show()

# Summary table
print(f"\n{'Run':<20s} {'Converged Epoch':<20s}")
print("-" * 40)
for run in sorted(all_runs, key=lambda x: (x['converged_epoch'] is None, x['converged_epoch'])):
    ep = run["converged_epoch"]
    print(f"{run['name']:<20s} {str(ep) if ep else 'Did not converge':<20s}")


Run                  Converged Epoch     
----------------------------------------
32 layers, residuals 105                 
32 layers, no residuals Did not converge    


In [ ]:
#@title smoke test helpers

import pandas as pd
from IPython.display import HTML
import random
import numpy as np
import torch

def generate_prediction_table(num_rows=10):
    html_output = []
    html_output.append("<table style='width:100%; border-collapse: collapse; text-align: center;'>")
    html_output.append("<thead><tr style='background-color:#f2f2f2;'>")
    html_output.append("<th style='padding: 8px; border: 1px solid #ddd;'>A</th>")
    html_output.append("<th style='padding: 8px; border: 1px solid #ddd;'>B</th>")
    html_output.append("<th style='padding: 8px; border: 1px solid #ddd;'>C</th>")
    html_output.append("<th style='padding: 8px; border: 1px solid #ddd;'>Math Formula</th>")
    html_output.append("<th style='padding: 8px; border: 1px solid #ddd;'>model Answer</th>")
    html_output.append("</tr></thead><tbody>")

    MODEL.eval()
    # Dynamically find the input dimension expected by the model
    c_classes = MODEL.proj_in.in_features - 200

    for _ in range(num_rows):
        # C is 1 to 19 inclusive
        a_rand = random.randint(0, 99)
        b_rand = random.randint(0, 99)
        c_rand = random.randint(1, 19)

        math_result = (a_rand * b_rand) % c_rand
        math_formula_str = f"({a_rand} * {b_rand}) % {c_rand}"

        # Encoding: c_rand-1 ensures correct index
        x_input = np.concatenate([
            one_hot(a_rand, num_classes=100),
            one_hot(b_rand, num_classes=100),
            one_hot(c_rand - 1, num_classes=c_classes)
        ])

        x_input_tensor = torch.tensor(x_input, dtype=torch.float32).to(DEVICE).unsqueeze(0)

        with torch.no_grad():
            logits = MODEL(x_input_tensor)
            predicted_class = logits.argmax(dim=1).item()

        is_correct = (predicted_class == math_result)
        model_answer_style = "background-color:#e6ffe6;" if is_correct else "background-color:#ffe6e6;"

        html_output.append(f"<tr>")
        html_output.append(f"<td style='padding: 8px; border: 1px solid #ddd;'>{a_rand}</td>")
        html_output.append(f"<td style='padding: 8px; border: 1px solid #ddd;'>{b_rand}</td>")
        html_output.append(f"<td style='padding: 8px; border: 1px solid #ddd;'>{c_rand}</td>")
        html_output.append(f"<td style='padding: 8px; border: 1px solid #ddd;'>{math_formula_str}</td>")
        html_output.append(f"<td style='padding: 8px; border: 1px solid #ddd; {model_answer_style}'>{predicted_class}</td>")
        html_output.append(f"</tr>")

    html_output.append("</tbody></table>")
    display(HTML("".join(html_output)))

generate_prediction_table()


A,B,C,Math Formula,Model Answer
81,14,1,(81 * 14) % 1,0
94,35,8,(94 * 35) % 8,2
28,17,4,(28 * 17) % 4,0
86,94,18,(86 * 94) % 18,2
11,75,14,(11 * 75) % 14,13
4,3,3,(4 * 3) % 3,0
27,29,17,(27 * 29) % 17,1
77,3,18,(77 * 3) % 18,15
25,91,18,(25 * 91) % 18,7
53,28,15,(53 * 28) % 15,14


# 5) Activation values visualized

In [ ]:
#@title Visualize the activation values at each layer
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.colors as mcolors
import torch

# Prefer the deployment font when available, otherwise fall back to common sans-serif fonts
font_candidates = ['Inter', 'DejaVu Sans', 'Arial', 'Helvetica', 'sans-serif']
plt.rcParams['font.family'] = font_candidates
font_path = 'fonts/Inter-Regular.ttf'
try:
    prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = prop.get_name()
except Exception:
    pass

# Set styling for all text and lines
plt.rcParams['text.color'] = '#F1F5F9'
plt.rcParams['axes.labelcolor'] = '#F1F5F9'
plt.rcParams['xtick.color'] = '#F1F5F9'
plt.rcParams['ytick.color'] = '#F1F5F9'
plt.rcParams['axes.edgecolor'] = '#F1F5F9'

# Grid styling
plt.rcParams['grid.color'] = '#5D6674'
plt.rcParams['grid.alpha'] = 0.7

def _coplot_luminance_from_rgba(rgba):
    r, g, b = mcolors.to_rgb(rgba)
    def _linear(c):
        return c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4
    return 0.2126 * _linear(r) + 0.7152 * _linear(g) + 0.0722 * _linear(b)

def _coplot_text_color_for_rgba(rgba, preferred='#F1F5F9'):
    try:
        preferred_l = _coplot_luminance_from_rgba(mcolors.to_rgba(preferred))
        bg_l = _coplot_luminance_from_rgba(rgba)
        lighter = max(preferred_l, bg_l)
        darker = min(preferred_l, bg_l)
        ratio = (lighter + 0.05) / (darker + 0.05)
        if ratio >= 4.5:
            return preferred
    except Exception:
        pass
    bg_l = _coplot_luminance_from_rgba(rgba)
    return '#111111' if bg_l > 0.5 else '#FFFFFF'

# ── Signal survival per layer (post-hoc) ──
xb, yb = next(iter(test_loader))

# Create figure and axis with publication-quality settings
fig, ax = plt.subplots(figsize=(10, 5.5), dpi=300)
fig.patch.set_facecolor('#1E293B')
ax.set_facecolor('#1E293B')

# Define distinct bright colors instead of the same blue, so lines are distinguishable
colors = ["#38BDF8", "#34D399", "#FBBF24", "#F87171", "#A78BFA"]
line_styles = ["-","--",":","-."]
marker_styles = ["o","s","^","D"]

for idx, run in enumerate(all_runs):
    model = run["model"].eval()
    if hasattr(model, "_orig_mod"):
        model = model._orig_mod

    magnitudes = []

    handles = []
    def make_hook(store):
        def hook(module, input, output):
            store.append(output.detach().abs().mean().item())
        return hook

    for block in model.layers:
        m = []
        magnitudes.append(m)
        handles.append(block.register_forward_hook(make_hook(m)))

    with torch.no_grad():
        model(xb)

    for h in handles:
        h.remove()

    layer_means = [m[0] for m in magnitudes]

    line_style = line_styles[idx % len(line_styles)] if False else '-'
    marker_style = marker_styles[idx % len(marker_styles)] if False else 'o'

    ax.plot(range(len(layer_means)), layer_means,
           color=colors[idx % len(colors)],
           linewidth=2,
           linestyle=line_style,
           marker=marker_style, markersize=5,
           label=run["name"],
           alpha=0.9)

ax.set_yscale("log")
ax.axhline(y=1.0, color="#F1F5F9", linestyle="--", alpha=0.3, linewidth=1)

# Customize axis labels
ax.set_xlabel('Layer index', fontsize=15, labelpad=10, color='#F1F5F9')
ax.set_ylabel('Activation value', fontsize=15, color='#F1F5F9', labelpad=10)
ax.set_title("Activation values in each layer", fontsize=16, color='#F1F5F9', pad=25)

# Add grid
ax.grid(axis='both', alpha=0.7, color='#5D6674', linestyle='--', linewidth=0.5, zorder=0)
ax.set_axisbelow(True)

# Add legend
ax.legend(loc='upper center',
         bbox_to_anchor=(0.5, 1.05),
         frameon=False, fontsize=12, ncol=3)

# Remove top and right spines for cleaner look
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)
ax.spines['bottom'].set_color('#F1F5F9')
ax.spines['left'].set_color('#F1F5F9')

# Adjust spine positions
ax.spines['left'].set_position(('outward', 2))
ax.spines['bottom'].set_position(('outward', 2))

# Tight layout
plt.tight_layout()

# Display the plot
plt.show()

# 6) 72 layers, no residual

In [ ]:
#assert False, "code up a DNN with 72 linear layers without using residuals"
MODEL = train(run_name="72 layers, have residuals", layer_depth=72, has_residuals=True)

  🚀 Run: 72 layers, have residuals
Epoch      1 | Train loss 2809507102.3023  acc 4.5% | Test loss 2810784374.0295  acc 4.4%
Epoch      2 | Train loss 1665641575.5857  acc 4.4% | Test loss 1663198209.7246  acc 4.4%
Epoch      3 | Train loss 992860274.2568  acc 3.1% | Test loss 989980772.0825  acc 3.1%
Epoch      4 | Train loss 658149303.3768  acc 7.9% | Test loss 654651451.9040  acc 8.0%
Epoch      5 | Train loss 461734211.7255  acc 9.4% | Test loss 459072584.0707  acc 9.1%
Epoch      6 | Train loss 307846981.4164  acc 7.6% | Test loss 305407151.2253  acc 7.7%
Epoch      7 | Train loss 210774787.0653  acc 4.5% | Test loss 208424709.5377  acc 4.4%
Epoch      8 | Train loss 133277028.8236  acc 2.9% | Test loss 130954904.5187  acc 2.9%
Epoch      9 | Train loss 89007244.5811  acc 33.8% | Test loss 87039100.3722  acc 33.6%
Epoch     10 | Train loss 55855848.2846  acc 7.6% | Test loss 54456190.2669  acc 7.7%
Epoch     11 | Train loss 49827124.7309  acc 6.6% | Test loss 48957920.3587  acc 6.

In [ ]:
#@title Answer
MODEL = train(run_name="72 layers, no residuals", layer_depth=72, has_residuals=False)

W1020 12:12:09.121000 1817 torch/_dynamo/convert_frame.py:1743] [0/8] torch._dynamo hit config.recompile_limit (8)
W1020 12:12:09.121000 1817 torch/_dynamo/convert_frame.py:1743] [0/8]    function: 'forward' (/tmp/ipykernel_1817/423275933.py:12)
W1020 12:12:09.121000 1817 torch/_dynamo/convert_frame.py:1743] [0/8]    last reason: 0/7: GLOBAL_STATE changed: grad_mode 
W1020 12:12:09.121000 1817 torch/_dynamo/convert_frame.py:1743] [0/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W1020 12:12:09.121000 1817 torch/_dynamo/convert_frame.py:1743] [0/8] To diagnose recompilation issues, see https://docs.pytorch.org/docs/main/user_guide/torch_compiler/compile/programming_model.recompilation.html


  🚀 Run: 72 layers, no residuals
Epoch      1 | Train loss 2.3858  acc 33.8% | Test loss 2.3937  acc 33.6%
Epoch      2 | Train loss 2.3535  acc 33.8% | Test loss 2.3594  acc 33.6%
Epoch      3 | Train loss 2.3522  acc 33.8% | Test loss 2.3576  acc 33.6%
Epoch      4 | Train loss 2.3537  acc 33.8% | Test loss 2.3591  acc 33.6%
Epoch      5 | Train loss 2.3525  acc 33.8% | Test loss 2.3575  acc 33.6%
Epoch      6 | Train loss 2.3530  acc 33.8% | Test loss 2.3589  acc 33.6%
Epoch      7 | Train loss 2.3521  acc 33.8% | Test loss 2.3576  acc 33.6%
Epoch      8 | Train loss 2.3521  acc 33.8% | Test loss 2.3573  acc 33.6%
Epoch      9 | Train loss 2.3521  acc 33.8% | Test loss 2.3574  acc 33.6%
Epoch     10 | Train loss 2.3518  acc 33.8% | Test loss 2.3569  acc 33.6%
Epoch     11 | Train loss 2.3518  acc 33.8% | Test loss 2.3570  acc 33.6%
Epoch     12 | Train loss 2.3514  acc 33.8% | Test loss 2.3567  acc 33.6%
Epoch     13 | Train loss 2.3518  acc 33.8% | Test loss 2.3573  acc 33.6%
Epoch

KeyboardInterrupt: 

# 7) DNN with 16 layers of 2xFFN growing & shrinking & ReLU

In [ ]:
assert False, "code up a DNN that has 16 layers of 2 FFN layers (growing and shrinking) alongside ReLU"

In [5]:
#@title Answer - DNN definition

class DNN(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, output_dim=OUTPUT_DIM, hidden_dim=128,
                 layer_depth=16, has_residuals=False):
        super().__init__()
        self.has_residuals = has_residuals
        self.proj_in = nn.Linear(input_dim, hidden_dim)
        self.layers = nn.ModuleList()
        for _ in range(layer_depth):
            self.layers.append(
                nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim * 2),
                    nn.ReLU(),
                    nn.Linear(hidden_dim * 2, hidden_dim))
                )
        self.proj_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = F.relu(self.proj_in(x))
        for layer in self.layers:
            if self.has_residuals:
                x = x + layer(x)
            else:
                x = layer(x)
        return self.proj_out(x)

MODEL = DNN().to(DEVICE)
print(MODEL)

print(f"Total parameters: {sum(p.numel() for p in MODEL.parameters()):,}")
print(f"Per-layer shapes:")
for name, param in MODEL.named_parameters():
    if "weight" in name:
        print(f"  {name:30s}  {str(list(param.shape)):>15s}  ({param.numel():>6,} params)")

DNN(
  (proj_in): Linear(in_features=219, out_features=128, bias=True)
  (layers): ModuleList(
    (0-15): 16 x Sequential(
      (0): Linear(in_features=128, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=128, bias=True)
    )
  )
  (proj_out): Linear(in_features=128, out_features=19, bias=True)
)
Total parameters: 1,085,331
Per-layer shapes:
  proj_in.weight                       [128, 219]  (28,032 params)
  layers.0.0.weight                    [256, 128]  (32,768 params)
  layers.0.2.weight                    [128, 256]  (32,768 params)
  layers.1.0.weight                    [256, 128]  (32,768 params)
  layers.1.2.weight                    [128, 256]  (32,768 params)
  layers.2.0.weight                    [256, 128]  (32,768 params)
  layers.2.2.weight                    [128, 256]  (32,768 params)
  layers.3.0.weight                    [256, 128]  (32,768 params)
  layers.3.2.weight                    [128, 256]  (32,768 params)
  l

In [ ]:
#@title answer - training
MODEL = train(run_name="16 layers of 2x growing & shrinking FFN, no residuals",
      layer_depth=16,
      has_residuals=False)

  🚀 Run: 16 layers of 2x growing & shrinking FFN, no residuals
Epoch      1 | Train loss 1.9674  acc 34.0% | Test loss 1.9782  acc 33.8%
Epoch      2 | Train loss 1.4956  acc 44.1% | Test loss 1.5157  acc 43.5%
Epoch      3 | Train loss 1.1711  acc 52.7% | Test loss 1.1854  acc 52.1%
Epoch      4 | Train loss 1.0773  acc 54.1% | Test loss 1.0803  acc 53.9%
Epoch      5 | Train loss 1.0499  acc 54.5% | Test loss 1.0557  acc 54.3%
Epoch      6 | Train loss 1.0590  acc 54.2% | Test loss 1.0627  acc 54.2%
Epoch      7 | Train loss 1.0782  acc 54.0% | Test loss 1.0816  acc 53.6%
Epoch      8 | Train loss 1.0392  acc 54.6% | Test loss 1.0446  acc 54.3%
Epoch      9 | Train loss 1.0536  acc 54.2% | Test loss 1.0544  acc 54.2%
Epoch     10 | Train loss 2.3398  acc 33.8% | Test loss 2.3470  acc 33.6%
Epoch     11 | Train loss 1.3061  acc 49.2% | Test loss 1.3108  acc 48.8%
Epoch     12 | Train loss 1.1322  acc 53.5% | Test loss 1.1379  acc 53.3%
Epoch     13 | Train loss 1.5789  acc 45.4% | Tes

In [6]:
MODEL = train(run_name="16 layers of 2x growing & shrinking FFN, with residuals", layer_depth=16, has_residuals=True)


  🚀 Run: 16 layers of 2x growing & shrinking FFN, with residuals


W1024 17:07:50.573000 8498 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


Epoch      1 | Train loss 1.0947  acc 54.1% | Test loss 1.0990  acc 53.9%
Epoch      2 | Train loss 1.0535  acc 54.6% | Test loss 1.0573  acc 54.4%
Epoch      3 | Train loss 1.0582  acc 54.5% | Test loss 1.0651  acc 54.2%
Epoch      4 | Train loss 1.0571  acc 54.5% | Test loss 1.0625  acc 54.3%
Epoch      5 | Train loss 1.0462  acc 54.5% | Test loss 1.0502  acc 54.3%
Epoch      6 | Train loss 0.9985  acc 57.5% | Test loss 1.0042  acc 57.0%
Epoch      7 | Train loss 0.8486  acc 64.9% | Test loss 0.8524  acc 64.8%
Epoch      8 | Train loss 0.7794  acc 67.6% | Test loss 0.7825  acc 67.2%
Epoch      9 | Train loss 0.7392  acc 72.1% | Test loss 0.7370  acc 72.0%
Epoch     10 | Train loss 0.6886  acc 72.6% | Test loss 0.6933  acc 72.2%
Epoch     11 | Train loss 0.6426  acc 74.3% | Test loss 0.6496  acc 73.9%
Epoch     12 | Train loss 0.5408  acc 78.4% | Test loss 0.5430  acc 78.4%
Epoch     13 | Train loss 0.5246  acc 79.2% | Test loss 0.5287  acc 79.3%
Epoch     14 | Train loss 0.5153  acc 

OptimizedModule(
  (_orig_mod): DNN(
    (proj_in): Linear(in_features=219, out_features=128, bias=True)
    (layers): ModuleList(
      (0-15): 16 x Sequential(
        (0): Linear(in_features=128, out_features=256, bias=True)
        (1): ReLU()
        (2): Linear(in_features=256, out_features=128, bias=True)
      )
    )
    (proj_out): Linear(in_features=128, out_features=19, bias=True)
  )
)

In [ ]:
#@title visualize activation values over layers
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.colors as mcolors
import torch

# Prefer the deployment font when available, otherwise fall back to common sans-serif fonts
font_candidates = ['Inter', 'DejaVu Sans', 'Arial', 'Helvetica', 'sans-serif']
plt.rcParams['font.family'] = font_candidates
font_path = 'fonts/Inter-Regular.ttf'
try:
    prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = prop.get_name()
except Exception:
    pass

# Set styling for all text and lines
plt.rcParams['text.color'] = '#F1F5F9'
plt.rcParams['axes.labelcolor'] = '#F1F5F9'
plt.rcParams['xtick.color'] = '#F1F5F9'
plt.rcParams['ytick.color'] = '#F1F5F9'
plt.rcParams['axes.edgecolor'] = '#F1F5F9'

# Grid styling
plt.rcParams['grid.color'] = '#5D6674'
plt.rcParams['grid.alpha'] = 0.7

def _coplot_luminance_from_rgba(rgba):
    r, g, b = mcolors.to_rgb(rgba)
    def _linear(c):
        return c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4
    return 0.2126 * _linear(r) + 0.7152 * _linear(g) + 0.0722 * _linear(b)

def _coplot_text_color_for_rgba(rgba, preferred='#F1F5F9'):
    try:
        preferred_l = _coplot_luminance_from_rgba(mcolors.to_rgba(preferred))
        bg_l = _coplot_luminance_from_rgba(rgba)
        lighter = max(preferred_l, bg_l)
        darker = min(preferred_l, bg_l)
        ratio = (lighter + 0.05) / (darker + 0.05)
        if ratio >= 4.5:
            return preferred
    except Exception:
        pass
    bg_l = _coplot_luminance_from_rgba(rgba)
    return '#111111' if bg_l > 0.5 else '#FFFFFF'

# ── Signal survival per layer (post-hoc) ──
xb, yb = next(iter(test_loader))

# Create figure and axis with publication-quality settings
fig, ax = plt.subplots(figsize=(10, 5.5), dpi=300)
fig.patch.set_facecolor('#1E293B')
ax.set_facecolor('#1E293B')

# Define distinct bright colors instead of the same blue, so lines are distinguishable
colors = ["#38BDF8", "#34D399", "#FBBF24", "#F87171", "#A78BFA"]
line_styles = ["-","--",":","-."]
marker_styles = ["o","s","^","D"]

for idx, run in enumerate(all_runs):
    model = run["model"].eval()
    if hasattr(model, "_orig_mod"):
        model = model._orig_mod

    magnitudes = []

    handles = []
    def make_hook(store):
        def hook(module, input, output):
            store.append(output.detach().abs().mean().item())
        return hook

    for block in model.layers:
        m = []
        magnitudes.append(m)
        handles.append(block.register_forward_hook(make_hook(m)))

    with torch.no_grad():
        model(xb)

    for h in handles:
        h.remove()

    layer_means = [m[0] for m in magnitudes]

    line_style = line_styles[idx % len(line_styles)] if False else '-'
    marker_style = marker_styles[idx % len(marker_styles)] if False else 'o'

    ax.plot(range(len(layer_means)), layer_means,
           color=colors[idx % len(colors)],
           linewidth=2,
           linestyle=line_style,
           marker=marker_style, markersize=5,
           label=run["name"],
           alpha=0.9)

ax.set_yscale("log")
ax.axhline(y=1.0, color="#F1F5F9", linestyle="--", alpha=0.3, linewidth=1)

# Customize axis labels
ax.set_xlabel('Layer index', fontsize=15, labelpad=10, color='#F1F5F9')
ax.set_ylabel('Activation value', fontsize=15, color='#F1F5F9', labelpad=10)
ax.set_title("Activation values in each layer", fontsize=16, color='#F1F5F9', pad=25)

# Add grid
ax.grid(axis='both', alpha=0.7, color='#5D6674', linestyle='--', linewidth=0.5, zorder=0)
ax.set_axisbelow(True)

# Add legend
ax.legend(loc='upper center',
         bbox_to_anchor=(0.5, 1.05),
         frameon=False, fontsize=12, ncol=3)

# Remove top and right spines for cleaner look
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)
ax.spines['bottom'].set_color('#F1F5F9')
ax.spines['left'].set_color('#F1F5F9')

# Adjust spine positions
ax.spines['left'].set_position(('outward', 2))
ax.spines['bottom'].set_position(('outward', 2))

# Tight layout
plt.tight_layout()

# Display the plot
plt.show()

In [ ]:
#@title Visualize results
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.colors as mcolors
import logging
import matplotlib.ticker as ticker

# Suppress missing font warnings
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

# Prefer the deployment font when available, otherwise fall back to common sans-serif fonts
font_candidates = ['Inter', 'DejaVu Sans', 'Arial', 'Helvetica', 'sans-serif']
plt.rcParams['font.family'] = font_candidates
font_path = 'fonts/Inter-Regular.ttf'
try:
    prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = prop.get_name()
except Exception:
    pass

# Set styling for all text and lines
plt.rcParams['text.color'] = '#F1F5F9'
plt.rcParams['axes.labelcolor'] = '#F1F5F9'
plt.rcParams['xtick.color'] = '#F1F5F9'
plt.rcParams['ytick.color'] = '#F1F5F9'
plt.rcParams['axes.edgecolor'] = '#F1F5F9'

# Grid styling
plt.rcParams['grid.color'] = '#5B7268'
plt.rcParams['grid.alpha'] = 0.7

def _coplot_luminance_from_rgba(rgba):
    r, g, b = mcolors.to_rgb(rgba)
    def _linear(c):
        return c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4
    return 0.2126 * _linear(r) + 0.7152 * _linear(g) + 0.0722 * _linear(b)

def _coplot_text_color_for_rgba(rgba, preferred='#F1F5F9'):
    try:
        preferred_l = _coplot_luminance_from_rgba(mcolors.to_rgba(preferred))
        bg_l = _coplot_luminance_from_rgba(rgba)
        lighter = max(preferred_l, bg_l)
        darker = min(preferred_l, bg_l)
        ratio = (lighter + 0.05) / (darker + 0.05)
        if ratio >= 4.5:
            return preferred
    except Exception:
        pass
    bg_l = _coplot_luminance_from_rgba(rgba)
    return '#111111' if bg_l > 0.5 else '#FFFFFF'

# Create figure and axis with publication-quality settings
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300)
fig.patch.set_facecolor('#1A3A2A')

# Distinct bright colors for dark background (modified to keep runs distinguishable)
colors = ["#38BDF8", "#34D399", "#FBBF24", "#F87171", "#A78BFA"]
line_styles = ["-","--",":","-."]
marker_styles = ["o","s","^","D"]

for idx, run in enumerate(all_runs):
    h = run["history"]

    line_style = line_styles[idx % len(line_styles)] if False else '-'
    marker_style = marker_styles[idx % len(marker_styles)] if False else ''

    # Plot accuracy
    axes[0].plot(h["epoch"], h["test_acc"], label=run["name"],
                 color=colors[idx % len(colors)], linewidth=2,
                 linestyle=line_style, marker=marker_style, alpha=0.9)

    # Plot loss
    axes[1].plot(h["epoch"], h["test_loss"], label=run["name"],
                 color=colors[idx % len(colors)], linewidth=2,
                 linestyle=line_style, marker=marker_style, alpha=0.9)

# Apply custom styling to both axes
for ax in axes:
    ax.set_facecolor('#1A3A2A')
    ax.set_xlabel('Epoch', fontsize=12, labelpad=10, color='#F1F5F9')

    # Force integer epochs
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

    # Add grid
    ax.grid(axis='both', alpha=0.7, color='#5B7268', linestyle='--', linewidth=0.5, zorder=0)
    ax.set_axisbelow(True)

    # Remove top and right spines for cleaner look
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(1)
    ax.spines['left'].set_linewidth(1)
    ax.spines['bottom'].set_color('#F1F5F9')
    ax.spines['left'].set_color('#F1F5F9')

    # Adjust spine positions
    ax.spines['left'].set_position(('outward', 2))
    ax.spines['bottom'].set_position(('outward', 2))

    # Add legend
    ax.legend(frameon=False, fontsize=10)

# Specific titles and labels
axes[0].set_ylabel('Test Accuracy', fontsize=12, color='#F1F5F9', labelpad=10)
axes[0].set_title('Test Accuracy', color='#F1F5F9', pad=15)
axes[0].yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1.0))

axes[1].set_ylabel('Test Loss', fontsize=12, color='#F1F5F9', labelpad=10)
axes[1].set_title('Test Loss', color='#F1F5F9', pad=15)
axes[1].set_yscale('linear')
axes[1].set_ylim(top=3.0)
axes[1].set_ylim(bottom=0.0)
axes[1].yaxis.set_major_formatter(ticker.ScalarFormatter())

# Global title
plt.suptitle("Residuals Comparison", fontsize=14, y=1.02, color='#F1F5F9')

# Tight layout
plt.tight_layout()
plt.show()

# Summary table
print(f"\n{'Run':<20s} {'Converged Epoch':<20s}")
print("-" * 40)
for run in sorted(all_runs, key=lambda x: (x['converged_epoch'] is None, x['converged_epoch'])):
    ep = run["converged_epoch"]
    print(f"{run['name']:<20s} {str(ep) if ep else 'Did not converge':<20s}")


Run                  Converged Epoch     
----------------------------------------
16 layers of 2x growing & shrinking FFN, with residuals 123                 
16 layers of 2x growing & shrinking FFN, no residuals Did not converge    


In [9]:
#@title (8) Save and load model
FILENAME = 'DDN-v3.pth'
MODEL = get_model('16 layers of 2x growing & shrinking FFN, with residuals')

print("--- Testing Local Save & Load ---")
load_model = DNN(layer_depth=16, has_residuals=True).to(DEVICE)
load_model = save_load_model_local(MODEL, load_model, filename_c=FILENAME)
predicted_value = invoke_model_with_a_b_c(load_model, 12, 1, 10)

--- Testing Local Save & Load ---
model weights saved locally to: .//DDN-v3.pth
model weights loaded successfully into a new model instance.
Prediction for (12 * 1) % 10 = 2


In [10]:
#@title (9) Save to Google Drive
pathname = save_to_google_drive(MODEL, FILENAME)

Model saved to Google Drive: /content/drive/MyDrive/workshop/DDN-v3.pth


In [12]:
#@title (10) Load Pretrained model from Hugging Face
print("\n--- Testing Hugging Face Download & Load ---")
hf_model = DNN(layer_depth=16, has_residuals=True).to(DEVICE)
hf_model = load_model_hugging_face(filename_c=FILENAME, new_model_c=hf_model)
predicted_value = invoke_model_with_a_b_c(hf_model, 12, 1, 10)

MODEL = hf_model



--- Testing Hugging Face Download & Load ---


DDN-v3.pth: reconstructing file:   0%|          |  0.00B / 4.36MB            

DDN-v3.pth: downloading bytes:           |  0.00B            

model downloaded from HF to: /root/.cache/huggingface/hub/models--howardyang2009--llm-workshop-v1/snapshots/c01a9309e30e876bfaebab7ee04d7d93b744e217/DDN-v3.pth
model weights loaded successfully into a new model instance.
Prediction for (12 * 1) % 10 = 2
